# 회의 녹취록 자동 생성기

**VibeVoice-ASR-7B** 기반 한국어/영어 회의 속기 시스템

## 사용 방법
1. Google Drive의 `회의녹음/input/` 폴더에 **.m4a 파일**을 넣으세요
2. 상단 메뉴에서 **런타임 > 모두 실행** 을 클릭하세요
3. 완료되면 `회의녹음/output/` 폴더에서 결과를 확인하세요

> 최초 실행 시 모델 다운로드에 약 10분이 소요됩니다. 이후 실행부터는 캐시를 사용합니다.

In [ ]:
#@title 1단계: 환경 설정 (최초 실행 시 약 3-5분)
import subprocess, sys, os

# GPU 확인
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "\u274c GPU가 필요합니다!\n"
        "상단 메뉴 > 런타임 > 런타임 유형 변경 > T4 GPU 선택 후 다시 실행하세요."
    )
gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"\u2705 GPU: {gpu_name} ({vram:.1f} GB)")

# ffmpeg 설치
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], capture_output=True)
print("\u2705 ffmpeg 설치 완료")

# VibeVoice 설치
if not os.path.exists("/content/VibeVoice"):
    print("VibeVoice를 설치합니다...")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/microsoft/VibeVoice.git", "/content/VibeVoice"],
        check=True
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "/content/VibeVoice"],
    check=True
)
print("\u2705 VibeVoice 설치 완료")

# voice_recog 레포 클론 (src/ 모듈 사용)
if not os.path.exists("/content/voice_recog"):
    subprocess.run(
        ["git", "clone", "-b", "dev",
         "https://github.com/gyusir/voice_recog.git", "/content/voice_recog"],
        check=True
    )
sys.path.insert(0, "/content/voice_recog/src")
print("\u2705 환경 설정 완료!")

In [ ]:
#@title 2단계: Google Drive 연결
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/회의녹음"
INPUT_DIR = os.path.join(BASE_DIR, "input")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
CACHE_DIR = os.path.join(BASE_DIR, "model_cache")
PROCESSED_FILE = os.path.join(BASE_DIR, ".processed")

for d in [INPUT_DIR, OUTPUT_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

m4a_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith('.m4a')]
print(f"\u2705 Drive 연결 완료!")
print(f"\u{1f4c1} input 폴더: {len(m4a_files)}개의 m4a 파일")

In [ ]:
#@title 3단계: 모델 로드
from pipeline import load_model
model, processor = load_model(CACHE_DIR)

In [ ]:
#@title 4단계: 녹취록 생성
from pipeline import run_pipeline
run_pipeline(INPUT_DIR, OUTPUT_DIR, CACHE_DIR, PROCESSED_FILE)
print("\n\u2705 모든 처리가 완료되었습니다!")
print(f"결과 확인: Google Drive > 회의녹음 > output 폴더")

---
## 문제 해결

| 문제 | 해결 방법 |
|---|---|
| GPU가 필요하다는 오류 | 런타임 > 런타임 유형 변경 > **T4 GPU** 선택 |
| GPU 메모리 부족 (OOM) | 런타임 > 런타임 다시 시작 후 재실행 |
| 파일이 처리되지 않음 | 파일 확장자가 **.m4a**인지 확인 |
| 이미 처리된 파일 재처리 | `회의녹음/.processed` 파일 삭제 후 재실행 |
| 모델 다운로드 중 끊김 | `회의녹음/model_cache/` 폴더 삭제 후 재실행 |